# LassoLars Regression on Walmart Weekly Sales - Technical Paper Version

This notebook implements a rigorous LassoLars Regression analysis suitable for academic/technical papers:
1) Loading and preprocessing the Walmart sales dataset
2) Feature engineering with domain knowledge
3) 60% validation, 20% training, 20% test split
4) Systematic hyperparameter optimization using cross-validation
5) Comprehensive evaluation on train, validation, and test sets
6) Metrics reported in interpretable units (dollars)

LassoLars Regression:
- Lasso with Least Angle Regression solver (efficient for high-dimensional sparse problems)
- Key hyperparameters: alpha (regularization strength)
- Hyperparameters optimized via 5-fold cross-validation
- Scaling recommended for linear models

In [ ]:
# 1) Imports and reproducibility
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.linear_model import LassoLars

np.random.seed(42)

In [ ]:
# 2) Load the dataset and parse dates safely
csv_candidates = [
    'walmart-sales-dataset-of-45stores.csv',
    '/Users/jaacabrera/Downloads/walmart-sales-dataset-of-45stores.csv',
]

df = None
for p in csv_candidates:
    try:
        df = pd.read_csv(p, low_memory=False)
        print(f'Loaded: {p}')
        break
    except Exception:
        pass

if df is None:
    raise FileNotFoundError('Could not find the dataset. Update the path in csv_candidates.')

df['Date'] = pd.to_datetime(df['Date'].astype('string'), errors='coerce', dayfirst=True)
bad_dates = df['Date'].isna().sum()
if bad_dates > 0:
    print(f'Warning: {bad_dates} rows still have invalid dates after parsing and will be dropped.')
    df = df.dropna(subset=['Date']).reset_index(drop=True)

if 'Weekly_Sales' not in df.columns:
    raise KeyError('Weekly_Sales column not found. Check your CSV headers.')
df = df.dropna(subset=['Weekly_Sales']).reset_index(drop=True)

print(f'Data shape after date/target checks: {df.shape}')
df.head(3)

In [ ]:
# 3) Feature engineering
df['Month'] = df['Date'].dt.month
df['DayOfWeek'] = df['Date'].dt.dayofweek
df['Week'] = df['Date'].dt.isocalendar().week.astype(int)
df['Quarter'] = df['Date'].dt.quarter
df['IsWeekend'] = (df['DayOfWeek'] >= 5).astype(int)

if 'Store' in df.columns:
    df['Store_Encoded'] = df['Store']
else:
    df['Store_Encoded'] = 0

feature_cols = ['Holiday_Flag','Temperature','Fuel_Price','CPI','Unemployment','Store_Encoded','Month','DayOfWeek','Week','Quarter','IsWeekend']
X = df[feature_cols].copy()
y = df['Weekly_Sales'].copy()
print('X shape:', X.shape, '| y shape:', y.shape)
print('Features:', feature_cols)

In [ ]:
# 4) Split 60/20/20
X_rem, X_test, y_rem, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_rem, y_rem, train_size=0.25, random_state=42)
print(f"Split sizes -> train: {len(X_train)} ({len(y_train)/len(y):.0%}), val: {len(X_val)} ({len(y_val)/len(y):.0%}), test: {len(X_test)} ({len(y_test)/len(y):.0%})")

In [ ]:
# 5) CV tuning (alpha) with scaling
imputer = SimpleImputer(strategy='median')
X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=feature_cols)
X_val_imp   = pd.DataFrame(imputer.transform(X_val), columns=feature_cols)
X_test_imp  = pd.DataFrame(imputer.transform(X_test), columns=feature_cols)

scaler = StandardScaler()
X_train_final = scaler.fit_transform(X_train_imp)
X_val_final   = scaler.transform(X_val_imp)
X_test_final  = scaler.transform(X_test_imp)

print('Performing 5-fold cross-validation to find optimal hyperparameters...')
print('Tuning: alpha')
print('This may take a few minutes (reduced grid for faster tuning)...\n')

param_grid = { 'alpha': [0.01, 0.1, 1.0] }

lassolars_cv = GridSearchCV(LassoLars(), param_grid=param_grid, cv=5, scoring='r2', n_jobs=-1, verbose=1)
lassolars_cv.fit(X_train_final, y_train)

model = lassolars_cv.best_estimator_
best_params = lassolars_cv.best_params_
cv_score = lassolars_cv.best_score_

print(f"\n{'='*70}")
print('CROSS-VALIDATION RESULTS')
print(f"{'='*70}")
print(f"Best alpha: {best_params['alpha']}")
print(f"CV R² score (5-fold): {cv_score:.4f}")
print(f"{'='*70}\n")

preds_train = model.predict(X_train_final)
preds_val = model.predict(X_val_final)
preds_test = model.predict(X_test_final)

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
rmse = lambda yt, yp: float(np.sqrt(mean_squared_error(yt, yp)))
metrics = {
    'TRAIN': {'R2': r2_score(y_train, preds_train), 'MSE': mean_squared_error(y_train, preds_train), 'RMSE': rmse(y_train, preds_train), 'MAE': mean_absolute_error(y_train, preds_train)},
    'VAL':   {'R2': r2_score(y_val, preds_val),     'MSE': mean_squared_error(y_val, preds_val),     'RMSE': rmse(y_val, preds_val),     'MAE': mean_absolute_error(y_val, preds_val)},
    'TEST':  {'R2': r2_score(y_test, preds_test),    'MSE': mean_squared_error(y_test, preds_test),    'RMSE': rmse(y_test, preds_test),    'MAE': mean_absolute_error(y_test, preds_test)}
}

print('LassoLars Regression Performance (hyperparameters optimized via CV)')
print(f"  Train Metrics       -> R²: {metrics['TRAIN']['R2']:.4f} | RMSE: ${metrics['TRAIN']['RMSE']:,.2f} | MAE: ${metrics['TRAIN']['MAE']:,.2f}")
print(f"  Validation Metrics  -> R²: {metrics['VAL']['R2']:.4f} | RMSE: ${metrics['VAL']['RMSE']:,.2f} | MAE: ${metrics['VAL']['MAE']:,.2f}")
print(f"  Test Metrics        -> R²: {metrics['TEST']['R2']:.4f} | RMSE: ${metrics['TEST']['RMSE']:,.2f} | MAE: ${metrics['TEST']['MAE']:,.2f}")
print(f"\n  ► Test set R² = {metrics['TEST']['R2']:.4f} (PRIMARY RESULT for papers)")

# Importances = |coefficients|
coef = getattr(model, 'coef_', np.zeros(len(feature_cols)))
feature_importance = pd.DataFrame({'Feature': feature_cols, 'Importance': np.abs(coef)}).sort_values('Importance', ascending=False)

# Exports
metrics_df = pd.DataFrame({'Dataset':['Train','Validation','Test'],'R²':[metrics['TRAIN']['R2'],metrics['VAL']['R2'],metrics['TEST']['R2']], 'MSE':[metrics['TRAIN']['MSE'],metrics['VAL']['MSE'],metrics['TEST']['MSE']], 'RMSE':[metrics['TRAIN']['RMSE'],metrics['VAL']['RMSE'],metrics['TEST']['RMSE']], 'MAE':[metrics['TRAIN']['MAE'],metrics['VAL']['MAE'],metrics['TEST']['MAE']]})
metrics_df.to_csv('LassoLars_Regression_Metrics.csv', index=False)

hyperparams_df = pd.DataFrame({'Hyperparameter':['alpha','cv_folds','cv_r2_score'], 'Value':[best_params['alpha'],5,cv_score], 'Tuned':['Yes (GridSearchCV)','N/A','N/A'], 'Description':['L1 regularization strength','Number of CV folds','Best R² across CV']})
hyperparams_df.to_csv('LassoLars_Hyperparameters.csv', index=False)

feature_importance.to_csv('LassoLars_Feature_Importances.csv', index=False)

train_results = pd.DataFrame({'Actual': y_train.values,'Predicted': preds_train,'Residual': y_train.values - preds_train,'Residual_Squared': (y_train.values - preds_train)**2,'Absolute_Error': np.abs(y_train.values - preds_train)})
train_results.to_csv('LassoLars_Train_Predictions.csv', index=False)
val_results = pd.DataFrame({'Actual': y_val.values,'Predicted': preds_val,'Residual': y_val.values - preds_val,'Residual_Squared': (y_val.values - preds_val)**2,'Absolute_Error': np.abs(y_val.values - preds_val)})
val_results.to_csv('LassoLars_Validation_Predictions.csv', index=False)
test_results = pd.DataFrame({'Actual': y_test.values,'Predicted': preds_test,'Residual': y_test.values - preds_test,'Residual_Squared': (y_test.values - preds_test)**2,'Absolute_Error': np.abs(y_test.values - preds_test)})
test_results.to_csv('LassoLars_Test_Predictions.csv', index=False)

print('✓ CSV exports written (metrics, hyperparameters, importances, predictions)')
print('
Excel formulas: MSE=AVERAGE(Residual_Squared), RMSE=SQRT(MSE), MAE=AVERAGE(Absolute_Error), R²=RSQ(Actual,Predicted)')

In [ ]:
# 6) Parity Plot: Actual vs Predicted
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(18,5))
axes[0].scatter(y_train, preds_train, alpha=0.5, edgecolors='k', linewidth=0.5, color='green'); axes[0].plot([y_train.min(),y_train.max()],[y_train.min(),y_train.max()],'r--',lw=2); axes[0].set_title(f"Train (R²={metrics['TRAIN']['R2']:.4f})")
axes[1].scatter(y_val, preds_val, alpha=0.5, edgecolors='k', linewidth=0.5); axes[1].plot([y_val.min(),y_val.max()],[y_val.min(),y_val.max()],'r--',lw=2); axes[1].set_title(f"Val (R²={metrics['VAL']['R2']:.4f})")
axes[2].scatter(y_test, preds_test, alpha=0.5, edgecolors='k', linewidth=0.5, color='orange'); axes[2].plot([y_test.min(),y_test.max()],[y_test.min(),y_test.max()],'r--',lw=2); axes[2].set_title(f"Test (R²={metrics['TEST']['R2']:.4f})")
for ax in axes: ax.set_xlabel('Actual'); ax.set_ylabel('Predicted'); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# 7) Model Configuration Summary
print('='*80)
print('LASSOLARS REGRESSION MODEL CONFIGURATION SUMMARY')
print('='*80)
print('
1. SELECTED FEATURES (11 total):')
print('-'*80)
for i, feat in enumerate(feature_cols, 1): print(f'   {i:2d}. {feat}')
print('
2. HYPERPARAMETERS (OPTIMIZED VIA CROSS-VALIDATION):')
print('-'*80)
print('   Model: LassoLars (sklearn.linear_model.LassoLars)')
print('   Search space: alpha ∈ [0.01, 0.1, 1.0]')
print(f"   Best alpha: {best_params['alpha']}")
print(f"   CV R² score: {cv_score:.4f}")
print('
   Preprocessing: SimpleImputer(median) + StandardScaler')
print('
3. FEATURE IMPORTANCES (|coefficients|):')
for idx, row in feature_importance.iterrows():
    bar_length = int((row['Importance'] / (feature_importance['Importance'].max()+1e-8)) * 50); bar='█'*bar_length
    print(f"      {row['Feature']:20s}: {row['Importance']:10.6f}  {bar}")
print('='*80)

# 8) Reporting Results in Technical Papers

- Model: LassoLars (linear; scaling required)
- Optimization: GridSearchCV (5-fold, scoring=R²); tuned alpha
- Use exported CSVs: LassoLars_Regression_Metrics.csv, LassoLars_Hyperparameters.csv, LassoLars_Feature_Importances.csv, predictions CSVs
- Report Test-set R² (primary), RMSE/MAE ($), parity plots, and coefficient-based importances.